# RAG Smoke Test

In [ ]:
pip install dotenv openai tiktoken "elasticsearch>=8,<9" psycopg2-binary

In [ ]:
import json

import sys
import os
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

# Ensure repo root is on path when running from notebooks/
repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Set DOCS_DIR relative to repo root if not already set
if not os.environ.get("DOCS_DIR"):
    os.environ["DOCS_DIR"] = str(repo_root / "docs")

## Indexing

In [ ]:
from rag.indexer import index_documents

index_documents()
print("Indexing complete")

## Dense Retrieval

In [ ]:
from rag.retriever import dense_retrieve

dense_results = dense_retrieve("transferencia internacional persona jurídica", top_k=3)
print(json.dumps(dense_results, indent=2, ensure_ascii=False))

## Sparse Retrieval

In [ ]:
from rag.retriever import sparse_retrieve

sparse_results = sparse_retrieve("transferencia internacional persona jurídica", top_k=3)
print(json.dumps(sparse_results, indent=2, ensure_ascii=False))

## Comparison

Una nota sobre la sección Comparison: dense_retrieve y sparse_retrieve devuelven list[str] según el tipo hint en retriever.py, así que no hay document_id explícito en los resultados. La celda de comparación usa extract_ids() que maneja ambos casos (dicts con clave document_id o strings directos), y hace el overlap con lo que haya. Si querés comparar por documento origen en lugar de por chunk, habría que cambiar las funciones del retriever para que devuelvan también el filename.

In [ ]:
from rag.retriever import dense_retrieve, sparse_retrieve

query = "persona políticamente expuesta PEP obligaciones reporte"

dense_res = dense_retrieve(query, top_k=5)
sparse_res = sparse_retrieve(query, top_k=5)

dense_ids = list(dense_res)
sparse_ids = list(sparse_res)

overlap = set(dense_ids) & set(sparse_ids)

print(f"Dense results : {len(dense_res)}")
print(f"Sparse results: {len(sparse_res)}")
print(f"Overlap       : {len(overlap)} — {overlap if overlap else 'none'}")